<a href="https://colab.research.google.com/github/ominivac/servidores_ce/blob/main/servidores_ce_08052026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Servidores Públicos do Ceará — Análise Completa
Fonte: https://github.com/ominivac/servidores_ce/tree/main/2025

## 1. Instalação de dependências

In [ ]:
# Execute apenas se necessário
# !pip install pandas requests

## 2. Configurações gerais

In [ ]:
REPO_API   = 'https://api.github.com/repos/ominivac/servidores_ce/contents/2025'
TETO_CONST = 22878.41

COLUMNS = [
    'nome', 'orgao', 'cargo', 'situacao',
    'total_descontos', 'abatimento_teto', 'outros_descontos',
    'sal_bruto', 'sal_liquido', 'diarias'
]

# Colunas que receberão sufixo de mês
COLS_MES = [
    'total_descontos', 'abatimento_teto', 'outros_descontos',
    'sal_bruto', 'sal_liquido', 'diarias'
]

# Colunas fixas (identificação do servidor)
COLS_FIXAS = ['orgao', 'cargo', 'situacao']

DTYPES = {
    'nome'            : 'str',
    'orgao'           : 'category',
    'cargo'           : 'category',
    'situacao'        : 'category',
    'total_descontos' : 'float32',
    'abatimento_teto' : 'float32',
    'outros_descontos': 'float32',
    'sal_bruto'       : 'float32',
    'sal_liquido'     : 'float32',
    'diarias'         : 'float32',
}

MESES_PT = {
    '01': 'jan', '02': 'fev', '03': 'mar', '04': 'abr',
    '05': 'mai', '06': 'jun', '07': 'jul', '08': 'ago',
    '09': 'set', '10': 'out', '11': 'nov', '12': 'dez'
}

## 3. Listar e baixar todos os CSVs do repositório

In [ ]:
resp = requests.get(REPO_API)
resp.raise_for_status()
arquivos = sorted([f for f in resp.json() if f['name'].endswith('.csv')], key=lambda x: x['name'])
print(f'{len(arquivos)} arquivo(s) CSV encontrado(s):')
for f in arquivos:
    print(' -', f['name'])

## 4. Concatenação horizontal (Join por `nome`)

Cada CSV representa um mês. As colunas numéricas recebem sufixo `_mmm` (ex: `sal_bruto_jan`).  
As colunas fixas (`orgao`, `cargo`, `situacao`) vêm do arquivo mais recente.

In [ ]:
frames_mes  = []   # DataFrames com colunas numéricas sufixadas
frames_fixo = []   # DataFrames com colunas fixas (um por mês, pegamos o último)

for arq in arquivos:
    # Extrai o mês do nome do arquivo  (ex: servidores_202501.csv → '01')
    match = re.search(r'(\d{4})(\d{2})', arq['name'])
    if not match:
        print(f'  ⚠ Não foi possível extrair mês de: {arq["name"]} — ignorado')
        continue

    ano, mes_num = match.group(1), match.group(2)
    sufixo = f"{MESES_PT.get(mes_num, mes_num)}_{ano}"   # ex: jan_2025

    print(f'Baixando: {arq["name"]}  →  sufixo: _{sufixo} ...', end=' ')

    raw = requests.get(arq['download_url']).content
    df_raw = pd.read_csv(
        io.BytesIO(raw),
        header=None,
        names=COLUMNS,
        dtype=DTYPES,
        engine='c',
        low_memory=True,
    )

    # Remove nome nulo/vazio e seta como índice
    df_raw = df_raw.dropna(subset=['nome'])
    df_raw = df_raw[df_raw['nome'].str.strip() != '']
    df_raw['nome'] = df_raw['nome'].str.strip().str.upper()
    df_raw = df_raw.set_index('nome')

    # 1) Colunas numéricas renomeadas com sufixo do mês
    df_mes = (
        df_raw[COLS_MES]
        .rename(columns={c: f'{c}_{sufixo}' for c in COLS_MES})
    )
    frames_mes.append(df_mes)

    # 2) Colunas fixas (mantemos a última versão disponível)
    frames_fixo.append(df_raw[COLS_FIXAS])

    print(f'{len(df_raw):,} registros')

# ── Junta colunas fixas (last-wins por nome) ──────────────────────
df_fixo = pd.concat(frames_fixo).groupby(level=0).last()

# ── Concatenação horizontal dos meses ─────────────────────────────
# outer join: servidor que não aparece num mês recebe NaN
df_wide = pd.concat(frames_mes, axis=1, join='outer')

# ── Une colunas fixas + colunas mensais ───────────────────────────
df_final = df_fixo.join(df_wide, how='outer')
df_final.index.name = 'nome'

print(f'\n✅ Dataset final: {df_final.shape[0]:,} servidores × {df_final.shape[1]} colunas')
print(f'   Memória       : {df_final.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

## 5. Prévia do dataset final

In [ ]:
df_final.head(5)

## 6. Colunas geradas

In [ ]:
print('\n'.join(df_final.columns.tolist()))

## 7. Salvar CSV consolidado (opcional)

In [ ]:
# Descomente para salvar
# df.to_csv('servidores_ce_2025_consolidado.csv')
# print('Arquivo salvo.')

## 7.1 Visão geral

In [ ]:
print('Shape :', df.shape)
print('Órgãos únicos :', df['orgao'].nunique())
print('Cargos únicos :', df['cargo'].nunique())
print()
print('Situações:')
print(df['situacao'].value_counts())
print()
print('Nulos por coluna:')
print(df.isnull().sum())

## 8. Análises sobre o dataset consolidado

In [ ]:
df[['sal_bruto','sal_liquido','total_descontos','diarias']].describe().round(2)

### 8.1 Evolução do sal_bruto mês a mês por servidor (Top 10 maiores)

In [ ]:
cols_bruto = [c for c in df_final.columns if c.startswith('sal_bruto_')]
df_final[cols_bruto].nlargest(10, cols_bruto[-1])

### 8.2 Servidores que aparecem em todos os meses

In [ ]:
cols_bruto = [c for c in df_final.columns if c.startswith('sal_bruto_')]
n_meses = len(cols_bruto)
presentes_sempre = df_final[cols_bruto].notna().sum(axis=1) == n_meses
print(f'Servidores presentes em todos os {n_meses} meses: {presentes_sempre.sum():,}')

### 8.3 Buscar servidor por nome

In [ ]:
busca = 'SERGIO LOBO'
resultado = df_final[df_final.index.str.contains(busca, case=False, na=False)]
print(f'{len(resultado)} registro(s) encontrado(s)')
resultado

### 8.4 Servidores acima do teto em qualquer mês

In [ ]:
cols_bruto = [c for c in df_final.columns if c.startswith('sal_bruto_')]
acima_teto = df_final[(df_final[cols_bruto] > TETO_CONST).any(axis=1)]
print(f'Servidores acima do teto (R$ {TETO_CONST:,.2f}) em pelo menos 1 mês: {len(acima_teto):,}')
acima_teto[['orgao','cargo','situacao'] + cols_bruto].sort_values(cols_bruto[-1], ascending=False).head(20)

## 8.5 Top 10 — Maiores salários brutos

In [ ]:
df.nlargest(10, 'sal_bruto')[['orgao','cargo','situacao','sal_bruto','sal_liquido']]

## 9. Média salarial por órgão (Top 15)

In [ ]:
(
    df.groupby('orgao', observed=True)[['sal_bruto','sal_liquido']]
    .mean()
    .round(2)
    .sort_values('sal_bruto', ascending=False)
    .head(15)
)

## 10. Média salarial por cargo (Top 15)

In [ ]:
(
    df.groupby('cargo', observed=True)[['sal_bruto','sal_liquido']]
    .mean()
    .round(2)
    .sort_values('sal_bruto', ascending=False)
    .head(15)
)

## 11. Maior diferença bruto entre cargos (Top 30)

In [ ]:
idx_min = df.groupby('cargo', observed=True)['sal_bruto'].idxmin()
idx_max = df.groupby('cargo', observed=True)['sal_bruto'].idxmax()

mins = df.loc[idx_min, ['cargo','orgao','sal_bruto']].rename(columns={'orgao':'orgao_min','sal_bruto':'sal_min'})
mins.index.name = 'nome_min'
mins = mins.reset_index()

maxs = df.loc[idx_max, ['cargo','orgao','sal_bruto']].rename(columns={'orgao':'orgao_max','sal_bruto':'sal_max'})
maxs.index.name = 'nome_max'
maxs = maxs.reset_index()

res = mins.merge(maxs, on='cargo')
res['diferenca'] = (res['sal_max'] - res['sal_min']).round(2)
res.sort_values('diferenca', ascending=False).head(30)

## 12. Maior diferença líquido entre cargos (Top 30)

In [ ]:
idx_min = df.groupby('cargo', observed=True)['sal_liquido'].idxmin()
idx_max = df.groupby('cargo', observed=True)['sal_liquido'].idxmax()

mins = df.loc[idx_min, ['cargo','orgao','sal_liquido']].rename(columns={'orgao':'orgao_min','sal_liquido':'liq_min'})
mins.index.name = 'nome_min'
mins = mins.reset_index()

maxs = df.loc[idx_max, ['cargo','orgao','sal_liquido']].rename(columns={'orgao':'orgao_max','sal_liquido':'liq_max'})
maxs.index.name = 'nome_max'
maxs = maxs.reset_index()

res = mins.merge(maxs, on='cargo')
res['diferenca'] = (res['liq_max'] - res['liq_min']).round(2)
res.sort_values('diferenca', ascending=False).head(30)

## 13. Servidores acima do teto constitucional

In [ ]:
acima = df[df['sal_bruto'] > TETO_CONST].sort_values('sal_bruto', ascending=False)
print(f'Servidores acima do teto (R$ {TETO_CONST:,.2f}): {len(acima):,}')
print()
print('Por situação:')
print(acima['situacao'].value_counts())
print()
print('Por órgão (Top 10):')
print(acima['orgao'].value_counts().head(10))

## 14. Buscar servidor por nome

In [ ]:
# Altere o nome abaixo para buscar qualquer servidor
busca = 'SERGIO LOBO'
resultado = df[df.index.str.contains(busca, case=False, na=False)]
print(f'{len(resultado)} registro(s) encontrado(s)')
resultado

## 15. Distribuição por faixa salarial

In [ ]:
bins   = [0, 2000, 5000, 10000, 20000, 40000, float('inf')]
labels = ['Até R$2k','R$2k-5k','R$5k-10k','R$10k-20k','R$20k-40k','Acima R$40k']
faixas = pd.cut(df['sal_bruto'], bins=bins, labels=labels)
faixas.value_counts().sort_index().to_frame('qtd')